# 01 - Apache Spark : Data Preprocessing

This notebook uses **Apache Spark (PySpark)** to load and clean the large traffic datasets:

| Dataset             | Rows        | Size  |
|---------------------|-------------|-------|
| `traffic_data.csv`  | 1,500,000   | ~105 MB |
| `area_monitor.csv`  | 1,000,000   | ~55 MB  |

Raw CSV files are too large for Pandas on a single machine - Spark distributes the
work across multiple CPU cores. The cleaned result is written to **Parquet**
(columnar, compressed) so the main analysis notebook can read it instantly.

> Run cells from top to bottom. The Spark session is created once and stopped at the end.


## 1. Spark Session Setup

`master("local[*]")` uses every available CPU core. A memory fraction is reserved so
the JVM does not get killed during shuffles.

In [1]:
# 1. Spark session setup
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf().setAppName("traffic_preprocessing").setMaster("local[*]")
conf.set("spark.sql.adaptive.enabled", "true")
conf.set("spark.sql.shuffle.partitions", "8")   # cores * 2 is a good default

spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")          # reduce log noise

print("Spark version   :", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/09 22:47:22 WARN Utils: Your hostname, pop-os, resolves to a loopback address: 127.0.0.1; using 10.162.216.80 instead (on interface wlp0s20f3)
26/08/09 22:47:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/heet18/Coding-Workspace/Futuristic/Heet/Github/Projects/Python-DataScience/myenv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/09 22:47:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where appl

Spark version   : 4.2.0
Default parallelism: 20


## 2. Load Raw CSVs

Spark reads the file lazily - no data is loaded until an **action** runs
(`show`, `count`, `write`, ...). We also inspect the inferred schema.

In [2]:
# 2. Resolve paths so the notebook works from any launch folder
import os
ROOT = os.path.abspath(".") if os.path.isdir("data") else os.path.abspath("..")
DATA_DIR = os.path.join(ROOT, "data")
print("Data directory:", DATA_DIR)

traffic_raw = spark.read.csv(
    os.path.join(DATA_DIR, "traffic_data.csv"), header=True, inferSchema=True,
)
monitor_raw = spark.read.csv(
    os.path.join(DATA_DIR, "area_monitor.csv"), header=True, inferSchema=True,
)

print("traffic_data rows  :", traffic_raw.count())
print("area_monitor rows  :", monitor_raw.count())

print("\n--- traffic_data schema ---")
traffic_raw.printSchema()
print("\n--- area_monitor schema ---")
monitor_raw.printSchema()

Data directory: /home/heet18/Coding-Workspace/Futuristic/Heet/Github/Projects/Python-DataScience/apache_spark/data


traffic_data rows  : 1500000
area_monitor rows  : 1000000

--- traffic_data schema ---
root
 |-- event_id: integer (nullable = true)
 |-- area_id: integer (nullable = true)
 |-- road_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- speed_kmh: double (nullable = true)
 |-- traffic_volume: integer (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- congestion_level: string (nullable = true)


--- area_monitor schema ---
root
 |-- monitor_id: integer (nullable = true)
 |-- area_id: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- sensor_type: string (nullable = true)
 |-- reading_value: double (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)



### 2.1 Peek at the Data

In [3]:
# Sample rows
traffic_raw.show(5, truncate=False)
print()
monitor_raw.show(5, truncate=False)

+--------+-------+-------+-------------------+----------+----+------------+---------+--------------+-----------------+----------------+
|event_id|area_id|road_id|timestamp          |date      |hour|vehicle_type|speed_kmh|traffic_volume|weather_condition|congestion_level|
+--------+-------+-------+-------------------+----------+----+------------+---------+--------------+-----------------+----------------+
|0       |9      |R009   |2024-03-07 04:10:31|2024-03-07|4   |truck       |41.6     |121           |cloudy           |low             |
|1       |78     |R019   |2024-03-20 09:32:36|2024-03-20|9   |truck       |67.4     |95            |clear            |low             |
|2       |66     |R011   |2024-02-21 11:01:04|2024-02-21|11  |auto        |86.1     |435           |rain             |high            |
|3       |44     |R014   |2024-02-07 06:18:58|2024-02-07|6   |bus         |33.3     |269           |clear            |medium          |
|4       |44     |R027   |2024-01-01 09:58:49|20

## 3. Data Quality Checks

### 3.1 Missing Values (per column)

In [4]:
# 3.1 Null counts across all columns (runs one job)
def null_summary(df, name):
    rows = df.select(
        [fn.sum(fn.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    )
    nulls = rows.collect()[0].asDict()
    print(f"--- {name} nulls ---")
    for col, cnt in nulls.items():
        print(f"  {col:20s}: {cnt:,}")
    return nulls

from pyspark.sql import functions as fn

t_nulls = null_summary(traffic_raw, "traffic_data")
m_nulls = null_summary(monitor_raw, "area_monitor")

--- traffic_data nulls ---
  event_id            : 0
  area_id             : 0
  road_id             : 0
  timestamp           : 0
  date                : 0
  hour                : 0
  vehicle_type        : 0
  speed_kmh           : 0
  traffic_volume      : 0
  weather_condition   : 0
  congestion_level    : 0
--- area_monitor nulls ---
  monitor_id          : 0
  area_id             : 0
  city                : 0
  sensor_type         : 0
  reading_value       : 0
  reading_timestamp   : 0
  status              : 0


### 3.2 Duplicate Records

In [5]:
# Count duplicates by the primary key
dup_t = traffic_raw.groupBy("event_id").count().filter("count > 1").count()
dup_m = monitor_raw.groupBy("monitor_id").count().filter("count > 1").count()
print("Duplicate event_id in traffic_data  :", dup_t)
print("Duplicate monitor_id in area_monitor:", dup_m)

# Drop any exact duplicate rows
traffic_raw = traffic_raw.dropDuplicates()
monitor_raw = monitor_raw.dropDuplicates()

Duplicate event_id in traffic_data  : 0
Duplicate monitor_id in area_monitor: 0


### 3.3 Range Check & Outliers (e.g. speed must be 0-150 km/h)

In [6]:
# Speed outliers outside a sane range
bad_speed = traffic_raw.filter(~fn.col("speed_kmh").between(0, 150)).count()
print("Speed values outside 0-150 km/h:", bad_speed)

# traffic_volume must be >= 0
bad_vol = traffic_raw.filter(fn.col("traffic_volume") < 0).count()
print("Negative traffic volumes       :", bad_vol)

Speed values outside 0-150 km/h: 0


Negative traffic volumes       : 0


## 4. Cleaning & Feature Engineering

We will:
1. Fill/remove nulls,
2. Cast types,
3. Add a `time_of_day` bucket and a `speed_category` label,
4. Drop columns we do not need.

In [7]:
# 4.1 Handle nulls & outliers
traffic = traffic_raw.filter(
    fn.col("speed_kmh").between(0, 150)
    & fn.col("traffic_volume").isNotNull()
    & fn.col("timestamp").isNotNull()
)
# Monitor: keep only active sensors
monitor = monitor_raw.filter(fn.col("status") == "active").drop("status")

print("After cleaning traffic rows:", traffic.count())
print("After cleaning monitor rows:", monitor.count())

After cleaning traffic rows: 1500000
After cleaning monitor rows: 750014


In [8]:
# 4.2 Feature engineering
traffic = (
    traffic
    .withColumn("date", fn.to_date("timestamp"))
    .withColumn("day_of_week", fn.date_format("timestamp", "EEEE"))
    .withColumn("time_of_day",
        fn.when(fn.hour("timestamp").between(0, 5), "night")
          .when(fn.hour("timestamp").between(6, 11), "morning")
          .when(fn.hour("timestamp").between(12, 16), "afternoon")
          .otherwise("evening"))
    .withColumn("speed_category",
        fn.when(fn.col("speed_kmh") < 20, "slow")
          .when(fn.col("speed_kmh") < 40, "moderate")
          .when(fn.col("speed_kmh") < 65, "normal")
          .otherwise("fast"))
)

traffic.select(
    "event_id", "timestamp", "date", "day_of_week",
    "time_of_day", "speed_kmh", "speed_category",
).show(8, truncate=False)

+--------+-------------------+----------+-----------+-----------+---------+--------------+
|event_id|timestamp          |date      |day_of_week|time_of_day|speed_kmh|speed_category|
+--------+-------------------+----------+-----------+-----------+---------+--------------+
|2       |2024-02-21 11:01:04|2024-02-21|Wednesday  |morning    |86.1     |fast          |
|20      |2024-03-11 07:23:35|2024-03-11|Monday     |morning    |72.9     |fast          |
|22      |2024-03-25 23:18:26|2024-03-25|Monday     |evening    |52.3     |normal        |
|28      |2024-02-22 15:29:31|2024-02-22|Thursday   |afternoon  |77.4     |fast          |
|44      |2024-03-11 03:53:00|2024-03-11|Monday     |night      |46.3     |normal        |
|50      |2024-01-01 05:49:56|2024-01-01|Monday     |night      |39.6     |moderate      |
|52      |2024-01-24 14:57:22|2024-01-24|Wednesday  |afternoon  |51.3     |normal        |
|62      |2024-02-09 04:09:12|2024-02-09|Friday     |night      |75.0     |fast          |

## 5. Descriptive Statistics

`summary()` gives min / max / mean / std on numeric columns - all distributed by Spark.

In [9]:
# 5. Summary statistics
traffic.select("speed_kmh", "traffic_volume").summary(
    "count", "mean", "stddev", "min", "max"
).show()

print()
traffic.groupBy("congestion_level").count().show()
traffic.groupBy("vehicle_type").count().show()

+-------+------------------+------------------+
|summary|         speed_kmh|    traffic_volume|
+-------+------------------+------------------+
|  count|           1500000|           1500000|
|   mean| 55.10431680000021|249.38559733333332|
| stddev|21.776057823896924| 144.3622640131606|
|    min|               5.0|                 0|
|    max|             130.0|               499|
+-------+------------------+------------------+




+----------------+------+
|congestion_level| count|
+----------------+------+
|             low|450767|
|            high|449420|
|          medium|599813|
+----------------+------+



+------------+------+
|vehicle_type| count|
+------------+------+
|        bike|300964|
|       truck|299552|
|         bus|299578|
|        auto|300956|
|         car|298950|
+------------+------+



## 6. Cache & Persist

The clean frame is **cached** in memory so repeated actions (multiple queries in the
next notebook) do not re-scan the 105 MB CSV every time.

In [10]:
# 6. Cache the cleaned traffic frame
traffic = traffic.cache()
print("Cached partitions:", traffic.rdd.getNumPartitions())

Cached partitions: 8


## 7. Write Cleaned Data to Parquet

Parquet is columnar + compressed -> much faster to re-read than CSV.

In [11]:
# 7. Persist cleaned data as parquet
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
traffic.write.mode("overwrite").parquet(os.path.join(PROCESSED_DIR, "traffic_clean.parquet"))
monitor.write.mode("overwrite").parquet(os.path.join(PROCESSED_DIR, "area_monitor_clean.parquet"))
print("Wrote parquet files under", PROCESSED_DIR)

26/08/09 22:48:08 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/08/09 22:48:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Wrote parquet files under /home/heet18/Coding-Workspace/Futuristic/Heet/Github/Projects/Python-DataScience/apache_spark/data/processed


## 8. Summary

- Loaded **2.5M** rows from CSV with Spark (no Pandas, no memory issues).
- Found & handled nulls, duplicates and speed outliers.
- Engineered `time_of_day`, `day_of_week`, `speed_category`.
- Cached in memory and persisted to Parquet.

The cleaned datasets are now ready for the main analysis notebook.

In [12]:
# 9. Stop the Spark session (releases JVM memory)
spark.stop()
print("Spark session stopped.")

Spark session stopped.
